# Comparison Results — All Methods on Test Chips
**NDWI | OWM | U-Net Best Model | U-Net Ensemble | Dummy Synthetic Masks**

Prerequisites before running:
- `ndwi_evaluation_final.ipynb` — completed ✅
- `generate_water_masks.py` — completed ✅
- `owm_scenes_final.ipynb` — completed ✅
- `unet_diagnostic_final.ipynb` — completed ✅ (provides final_summary.csv, prob_maps_best/, prob_maps_ensemble/)

U-Net inference and threshold selection are handled entirely by unet_diagnostic_final.ipynb.
This notebook loads those results and produces the final comparison figures.

In [42]:
# ── Cell 1: Setup ──
import json
import numpy as np
import rasterio
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from pathlib import Path
from collections import defaultdict
from sklearn.metrics import f1_score, jaccard_score, precision_score, recall_score
from tqdm import tqdm

BASE_DIR   = Path('/mnt/batch/tasks/shared/LS_root/mounts/clusters/v-lmarotti1/code/Users/v-lmarotti/OmniWaterMask')
CHIPS_DIR  = BASE_DIR / 'images/chips_images/images'
MASKS_DIR  = BASE_DIR / 'images/chips_masks/masks'
NDWI_DIR   = BASE_DIR / 'results/ndwi_chips'
NDWI_MASKS_DIR    = BASE_DIR / 'results/ndwi_chips/masks_t0.35'
OWM_DIR           = BASE_DIR / 'results/owm_chips'
DUMMY_DIR         = BASE_DIR / 'results/dummy_masks'
DIAG_DIR          = BASE_DIR / 'results/unet_diagnostic_v3'
PROB_DIR_BEST     = DIAG_DIR / 'prob_maps_best'
PRED_DIR_MAJORITY = DIAG_DIR / 'pred_maps_ensemble_majority'
FINAL_SUMMARY_PATH = DIAG_DIR / 'final_summary.csv'
OUTPUT_DIR        = BASE_DIR / 'results/comparison_all'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NDWI_THRESH = 0.35  # dev threshold — honest operational estimate
THRESH_ENS  = 'majority vote'

# Load test chips
with open(BASE_DIR / 'results/unet_training/test_chips.json') as f:
    test_chip_names = json.load(f)
test_chips = [CHIPS_DIR / n for n in test_chip_names]

def get_scene_id(filename):
    name = Path(filename).stem
    if '_clipped_' in name: return name.split('_clipped_')[0]
    if '_clip_' in name:    return name.split('_clip_')[0]
    return name

# Load U-Net results from unet_diagnostic_final.ipynb
if not FINAL_SUMMARY_PATH.exists():
    raise FileNotFoundError(f'final_summary.csv not found — run unet_diagnostic_final.ipynb first.')
unet_summary  = pd.read_csv(FINAL_SUMMARY_PATH)
unet_best_row = unet_summary[unet_summary['method'] == 'U-Net best model'].iloc[0]
unet_ens_row  = unet_summary[unet_summary['method'] == 'U-Net ensemble (majority vote)'].iloc[0]
THRESH_BEST   = float(unet_best_row['threshold'])

print(f'Test chips:          {len(test_chips):,}')
print(f'U-Net best model:    t={THRESH_BEST}, F1={unet_best_row["micro_f1"]:.4f}')
print(f'U-Net ensemble:      majority vote, F1={unet_ens_row["micro_f1"]:.4f}')
print(f'NDWI threshold:      t={NDWI_THRESH} (dev threshold)')
print(f'\nResults availability:')
print(f'  NDWI sweep:        {(NDWI_DIR / "ndwi_micro_sweep.csv").exists()}')
print(f'  OWM predictions:   {len(list(OWM_DIR.glob("*.tif"))) if OWM_DIR.exists() else 0} files')
print(f'  Prob maps best:    {len(list(PROB_DIR_BEST.glob("*.tif")))} files')

Test chips:          4,788
U-Net best model:    t=0.25, F1=0.8413
U-Net ensemble:      majority vote, F1=0.8601
NDWI threshold:      t=0.35 (dev threshold)

Results availability:
  NDWI sweep:        True
  OWM predictions:   4788 files
  Prob maps best:    4788 files


In [55]:
# ── Cell 1b: Generate NDWI masks at t=0.35 from raw chips ──
# Recomputes NDWI from raw imagery and saves binary masks at t=0.35.
# Only needs to run once — skips chips that already exist.

NDWI_MASKS_DIR = BASE_DIR / 'results/ndwi_chips/masks_t0.35'
NDWI_MASKS_DIR.mkdir(parents=True, exist_ok=True)

existing   = set(f.name for f in NDWI_MASKS_DIR.glob('*.tif'))
to_process = [c for c in test_chips if c.name not in existing]
print(f'Chips to process: {len(to_process)} (already done: {len(existing)})')

for chip_path in tqdm(to_process, desc='Generating NDWI masks t=0.35'):
    mask_path = MASKS_DIR / chip_path.name
    if not chip_path.exists() or not mask_path.exists(): continue
    with rasterio.open(chip_path) as src:
        green = src.read(3).astype(np.float32)
        nir1  = src.read(7).astype(np.float32)
        profile = src.profile.copy()
    ndwi = (green - nir1) / (green + nir1 + 1e-6)
    mask = (ndwi >= 0.35).astype(np.uint8)
    profile.update(count=1, dtype='uint8', nodata=None)
    with rasterio.open(NDWI_MASKS_DIR / chip_path.name, 'w', **profile) as dst:
        dst.write(mask[np.newaxis, :, :])

print(f'Done — masks saved to {NDWI_MASKS_DIR}')

Chips to process: 4788 (already done: 0)


Generating NDWI masks t=0.35: 100%|██████████| 4788/4788 [23:17<00:00,  3.43it/s]

Done — masks saved to /mnt/batch/tasks/shared/LS_root/mounts/clusters/v-lmarotti1/code/Users/v-lmarotti/OmniWaterMask/results/ndwi_chips/masks_t0.35


In [58]:
# ── Cell 2: NDWI Setup ──
# Chip-level CSV does not contain t=0.35 — masks generated in Cell 1b instead.
# df_ndwi set to None; NDWI metrics computed directly from masks_t0.35/ in Cell 5.

NDWI_THRESH = 0.35
df_ndwi = None
print(f'NDWI threshold: t={NDWI_THRESH} (dev threshold)')
print('Chip-level CSV not used — NDWI masks at t=0.35 generated in Cell 1b.')

NDWI threshold: t=0.35 (dev threshold)
Chip-level CSV not used — NDWI masks at t=0.35 generated in Cell 1b.


In [59]:
# ── Cell 3: Load OWM Results ──
# First run: loads from raw prediction files and saves cache.
# Subsequent runs: loads from cache instantly.

CACHE_PATH = OUTPUT_DIR / 'df_owm.csv'

if CACHE_PATH.exists():
    df_owm = pd.read_csv(CACHE_PATH)
    print(f'Loaded df_owm from cache ({len(df_owm):,} rows)')
else:
    owm_files = sorted(OWM_DIR.glob('*.tif')) if OWM_DIR.exists() else []
    if not owm_files:
        print('OWM predictions not found — run owm_scenes_final.ipynb first.')
        df_owm = None
    else:
        print(f'OWM predictions: {len(owm_files)} files — loading...')
        owm_rows = []
        all_preds, all_truths = [], []
        for chip_path in tqdm(test_chips, desc='Loading OWM'):
            pred_path = OWM_DIR / chip_path.name
            mask_path = MASKS_DIR / chip_path.name
            if not pred_path.exists() or not mask_path.exists(): continue
            with rasterio.open(pred_path) as src: pred  = (src.read(1) > 0).astype(np.uint8)
            with rasterio.open(mask_path) as src: truth = (src.read(1) > 0).astype(np.uint8)
            all_preds.append(pred.flatten()); all_truths.append(truth.flatten())
            owm_rows.append({
                'chip':      chip_path.name,
                'scene':     get_scene_id(chip_path.name),
                'f1':        round(f1_score(truth.flatten(), pred.flatten(), zero_division=0), 4),
                'iou':       round(jaccard_score(truth.flatten(), pred.flatten(), zero_division=0), 4),
                'precision': round(precision_score(truth.flatten(), pred.flatten(), zero_division=0), 4),
                'recall':    round(recall_score(truth.flatten(), pred.flatten(), zero_division=0), 4),
            })
        df_owm = pd.DataFrame(owm_rows)
        p = np.concatenate(all_preds); t = np.concatenate(all_truths)
        print(f'OWM micro F1={f1_score(t,p,zero_division=0):.4f}, P={precision_score(t,p,zero_division=0):.4f}, R={recall_score(t,p,zero_division=0):.4f}')
        df_owm.to_csv(CACHE_PATH, index=False)
        print(f'Cached to {CACHE_PATH}')

Loaded df_owm from cache (4,788 rows)


In [60]:
# ── Cell 4: Load Dummy Mask Results ──
# First run: computes from raw files and saves cache.
# Subsequent runs: loads from cache instantly.

import json as json_module
CACHE_PATH   = OUTPUT_DIR / 'dummy_results.json'
WATER_RATIOS = [0, 20, 40, 50, 60, 80, 100]

if CACHE_PATH.exists():
    with open(CACHE_PATH) as f:
        dummy_results = json_module.load(f)
    print(f'Loaded dummy_results from cache ({len(dummy_results)} configurations)')
    for pct, metrics in dummy_results.items():
        print(f'  Dummy {pct}%: F1={metrics["micro_f1"]:.4f}')
else:
    dummy_results = {}
    for pct in WATER_RATIOS:
        ratio_dir = DUMMY_DIR / f'water_{pct:03d}pct'
        if not ratio_dir.exists():
            print(f'Dummy {pct}%: not found — skipping'); continue
        all_preds, all_truths = [], []
        for chip_path in tqdm(test_chips, desc=f'Dummy {pct}%', leave=False):
            dummy_path = ratio_dir / chip_path.name
            mask_path  = MASKS_DIR / chip_path.name
            if not dummy_path.exists() or not mask_path.exists(): continue
            with rasterio.open(dummy_path) as src: pred  = (src.read(1) > 0).astype(np.uint8)
            with rasterio.open(mask_path)  as src: truth = (src.read(1) > 0).astype(np.uint8)
            all_preds.append(pred.flatten()); all_truths.append(truth.flatten())
        p = np.concatenate(all_preds); t = np.concatenate(all_truths)
        dummy_results[str(pct)] = {
            'micro_f1':  round(f1_score(t, p, zero_division=0), 4),
            'precision': round(precision_score(t, p, zero_division=0), 4),
            'recall':    round(recall_score(t, p, zero_division=0), 4),
            'iou':       round(jaccard_score(t, p, zero_division=0), 4),
        }
        print(f'Dummy {pct}%: F1={dummy_results[str(pct)]["micro_f1"]:.4f}')
    with open(CACHE_PATH, 'w') as f:
        json_module.dump(dummy_results, f)
    print(f'Cached to {CACHE_PATH}')

Loaded dummy_results from cache (7 configurations)
  Dummy 0%: F1=0.0000
  Dummy 20%: F1=0.0431
  Dummy 40%: F1=0.0456
  Dummy 50%: F1=0.0461
  Dummy 60%: F1=0.0465
  Dummy 80%: F1=0.0469
  Dummy 100%: F1=0.0472


In [ ]:
# ── Cell 5: Summary Comparison Table ──
# Loads U-Net results directly from unet_diagnostic_final.ipynb output.
# Recomputes NDWI and OWM micro metrics from prediction files for consistency.

rows = []

# NDWI — computed directly from masks_t0.35/
all_p, all_t = [], []
for c in test_chips:
    mp = NDWI_MASKS_DIR / c.name
    if not mp.exists(): continue
    with rasterio.open(mp) as src: all_p.append(src.read(1).flatten())
    with rasterio.open(MASKS_DIR/c.name) as src: all_t.append((src.read(1)>0).astype(np.uint8).flatten())
p = np.concatenate(all_p); t = np.concatenate(all_t)
rows.append({'Method': f'NDWI (t={NDWI_THRESH})',
             'Micro F1':  round(f1_score(t,p,zero_division=0),4),
             'Precision': round(precision_score(t,p,zero_division=0),4),
             'Recall':    round(recall_score(t,p,zero_division=0),4),
             'IoU':       round(jaccard_score(t,p,zero_division=0),4)})

# OWM
if df_owm is not None:
    all_p, all_t = [], []
    for c in test_chips:
        op = OWM_DIR / c.name
        if not op.exists(): continue
        with rasterio.open(op) as src: all_p.append((src.read(1)>0).astype(np.uint8).flatten())
        with rasterio.open(MASKS_DIR/c.name) as src: all_t.append((src.read(1)>0).astype(np.uint8).flatten())
    p = np.concatenate(all_p); t = np.concatenate(all_t)
    rows.append({'Method': 'OWM',
                 'Micro F1':  round(f1_score(t,p,zero_division=0),4),
                 'Precision': round(precision_score(t,p,zero_division=0),4),
                 'Recall':    round(recall_score(t,p,zero_division=0),4),
                 'IoU':       round(jaccard_score(t,p,zero_division=0),4)})

# U-Net best model — from unet_diagnostic_final.ipynb
rows.append({'Method': f'U-Net best model (t={THRESH_BEST})',
             'Micro F1':  unet_best_row['micro_f1'],
             'Precision': unet_best_row['precision'],
             'Recall':    unet_best_row['recall'],
             'IoU':       None})

# U-Net ensemble — from unet_diagnostic_final.ipynb
rows.append({'Method': f'U-Net ensemble (t={THRESH_ENS})',
             'Micro F1':  unet_ens_row['micro_f1'],
             'Precision': unet_ens_row['precision'],
             'Recall':    unet_ens_row['recall'],
             'IoU':       None})

# Dummy sanity checks
for pct in [0, 100]:
    if str(pct) in dummy_results:
        m = dummy_results[str(pct)]
        rows.append({'Method': f'Dummy {pct}% water',
                     'Micro F1': m['micro_f1'], 'Precision': m['precision'],
                     'Recall': m['recall'], 'IoU': m['iou']})

summary_df = pd.DataFrame(rows).sort_values('Micro F1', ascending=False)
summary_df.to_csv(OUTPUT_DIR / 'final_comparison.csv', index=False)

print('\n' + '='*70)
print('COMPARISON — Micro F1 (all pixels pooled across 4,788 test chips)')
print('='*70)
print(summary_df.to_string(index=False))

In [ ]:
# ── Cell 6: Bar Charts ──

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
COLORS = ['#2E75B6', '#5DCAA5', '#378ADD', '#1D9E75', '#B4B2A9', '#C8C8C8']

for ax, metric in zip(axes, ['Micro F1', 'Precision', 'Recall', 'IoU']):
    vals = summary_df[metric].fillna(0)
    bars = ax.bar(summary_df['Method'], vals, color=COLORS[:len(summary_df)])
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    ax.set_title(metric); ax.set_ylim(0, 1.15)
    ax.set_xticklabels(summary_df['Method'], rotation=25, ha='right', fontsize=7)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Micro Metrics — All Methods on 4,788 Test Chips', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'comparison_bar_charts.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 7: Per-Scene F1 Heatmap ──
# Best model uses probability maps + threshold.
# Ensemble uses majority vote binary predictions.

def per_scene_micro_prob(prob_dir, thresh, label):
    scene_map = defaultdict(list)
    for c in test_chips:
        scene_map[get_scene_id(c.name)].append(c)
    rows = {}
    for scene_id, chips in scene_map.items():
        all_p, all_t = [], []
        for cp in chips:
            pp = prob_dir / cp.name
            mp = MASKS_DIR / cp.name
            if not pp.exists(): continue
            with rasterio.open(pp) as src: prob = src.read(1)
            with rasterio.open(mp) as src: truth = (src.read(1) > 0).astype(np.uint8)
            all_p.append((prob >= thresh).astype(np.uint8).flatten())
            all_t.append(truth.flatten())
        if all_p:
            p = np.concatenate(all_p); t = np.concatenate(all_t)
            rows[scene_id] = round(f1_score(t, p, zero_division=0), 4)
    return pd.Series(rows, name=label)

def per_scene_micro_binary(pred_dir, label):
    scene_map = defaultdict(list)
    for c in test_chips:
        scene_map[get_scene_id(c.name)].append(c)
    rows = {}
    for scene_id, chips in scene_map.items():
        all_p, all_t = [], []
        for cp in chips:
            pp = pred_dir / cp.name
            mp = MASKS_DIR / cp.name
            if not pp.exists(): continue
            with rasterio.open(pp) as src: pred = src.read(1).flatten()
            with rasterio.open(mp) as src: truth = (src.read(1) > 0).astype(np.uint8).flatten()
            all_p.append(pred)
            all_t.append(truth)
        if all_p:
            p = np.concatenate(all_p); t = np.concatenate(all_t)
            rows[scene_id] = round(f1_score(t, p, zero_division=0), 4)
    return pd.Series(rows, name=label)

scene_data = {}
if df_ndwi is not None:
    scene_data[f'NDWI (t={NDWI_THRESH})'] = df_ndwi.groupby('scene')['f1'].mean().round(4)
if df_owm is not None:
    scene_data['OWM'] = df_owm.groupby('scene')['f1'].mean().round(4)

scene_data[f'U-Net best (t={THRESH_BEST})'] = per_scene_micro_prob(
    PROB_DIR_BEST, THRESH_BEST, f'best_t{THRESH_BEST}')
scene_data['U-Net ensemble (majority vote)'] = per_scene_micro_binary(
    PRED_DIR_MAJORITY, 'ensemble_majority')

scene_df = pd.DataFrame(scene_data).fillna(0)
scene_df.to_csv(OUTPUT_DIR / 'per_scene_comparison.csv')

fig, ax = plt.subplots(figsize=(max(12, len(scene_data)*3), len(scene_df)*0.5+2))
im = ax.imshow(scene_df.values, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(scene_df.columns)))
ax.set_xticklabels(scene_df.columns, fontsize=9)
ax.set_yticks(range(len(scene_df)))
ax.set_yticklabels([s[:35] for s in scene_df.index], fontsize=7)
for i in range(len(scene_df)):
    for j in range(len(scene_df.columns)):
        val = scene_df.values[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=7, color='black' if 0.3 < val < 0.8 else 'white')
plt.colorbar(im, ax=ax, label='Micro F1')
plt.title('Per-Scene Micro F1 — All Methods', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'per_scene_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 8: Visual Inspection — All 13 Test Scenes ──
# Shows RGB, ground truth, NDWI, OWM, U-Net best model, U-Net ensemble (majority vote).

VISUALS_DIR = OUTPUT_DIR / 'visuals'
VISUALS_DIR.mkdir(exist_ok=True)
TEST_SCENES = sorted(set(get_scene_id(c.name) for c in test_chips))
print(f'Generating visuals for {len(TEST_SCENES)} scenes...')

for scene_id in TEST_SCENES:
    scene_chips = [c for c in test_chips if get_scene_id(c.name) == scene_id]
    positions   = []
    for c in scene_chips:
        parts = c.stem.split('_')
        try:
            row = int([p for p in parts if p.startswith('row')][0].replace('row',''))
            col = int([p for p in parts if p.startswith('col')][0].replace('col',''))
            positions.append(((row, col), c))
        except: continue
    if not positions:
        print(f'  Skipping {scene_id} — could not parse row/col')
        continue

    max_row = max(r for (r,_),_ in positions) + 512
    max_col = max(c for (_,c),_ in positions) + 512
    rgb_c    = np.zeros((max_row, max_col, 3), dtype=np.uint8)
    truth_c  = np.zeros((max_row, max_col), dtype=np.uint8)
    ndwi_c   = np.zeros((max_row, max_col), dtype=np.uint8)
    owm_c    = np.zeros((max_row, max_col), dtype=np.uint8)
    best_c   = np.zeros((max_row, max_col), dtype=np.uint8)
    ens_c    = np.zeros((max_row, max_col), dtype=np.uint8)

    for (row, col), cp in positions:
        with rasterio.open(cp) as src:
            def s(a):
                p2, p98 = np.percentile(a, 2), np.percentile(a, 98)
                return np.clip((a-p2)/(p98-p2+1e-6)*255, 0, 255).astype(np.uint8)
            rgb_c[row:row+512, col:col+512, 0] = s(src.read(5).astype(np.float32))
            rgb_c[row:row+512, col:col+512, 1] = s(src.read(3).astype(np.float32))
            rgb_c[row:row+512, col:col+512, 2] = s(src.read(2).astype(np.float32))
        with rasterio.open(MASKS_DIR/cp.name) as src:
            truth_c[row:row+512, col:col+512] = (src.read(1) > 0)
        np_path = NDWI_MASKS_DIR / cp.name
        if np_path.exists():
            with rasterio.open(np_path) as src: ndwi_c[row:row+512, col:col+512] = src.read(1)
        op = OWM_DIR / cp.name
        if op.exists():
            with rasterio.open(op) as src: owm_c[row:row+512, col:col+512] = (src.read(1) > 0)
        bp = PROB_DIR_BEST / cp.name
        if bp.exists():
            with rasterio.open(bp) as src: best_c[row:row+512, col:col+512] = (src.read(1) >= THRESH_BEST)
        ep = PRED_DIR_MAJORITY / cp.name
        if ep.exists():
            with rasterio.open(ep) as src: ens_c[row:row+512, col:col+512] = src.read(1)

    panels = [
        (rgb_c,   'RGB',                             None),
        (truth_c, 'Ground Truth',                    'Blues'),
        (ndwi_c,  f'NDWI (t={NDWI_THRESH})',         'Blues'),
        (owm_c,   'OWM',                             'Blues'),
        (best_c,  f'U-Net best (t={THRESH_BEST})',    'Blues'),
        (ens_c,   'U-Net ensemble (majority vote)',   'Blues'),
    ]
    fig, axes = plt.subplots(1, 6, figsize=(30, 6))
    for ax, (data, title, cmap) in zip(axes, panels):
        ax.imshow(data, cmap=cmap); ax.set_title(title, fontsize=8); ax.axis('off')
    plt.suptitle(scene_id, fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig(VISUALS_DIR / f'{scene_id}.png', dpi=80, bbox_inches='tight')
    plt.close()
    print(f'  Saved: {scene_id}')

print(f'Done — {len(TEST_SCENES)} scenes saved to {VISUALS_DIR}')

In [ ]:
# ── Cell 9: Confusion Maps — All 13 Test Scenes ──
# TP=blue, FP=red, FN=orange, TN=white
# Shows NDWI, U-Net best model, and U-Net ensemble (majority vote).

CONF_DIR = OUTPUT_DIR / 'confusion_maps'
CONF_DIR.mkdir(exist_ok=True)
CMAP   = ListedColormap(['white', 'steelblue', 'crimson', 'darkorange'])
LEGEND = [
    mpatches.Patch(color='white',      label='TN — correct land'),
    mpatches.Patch(color='steelblue',  label='TP — correct water'),
    mpatches.Patch(color='crimson',    label='FP — false water'),
    mpatches.Patch(color='darkorange', label='FN — missed water'),
]

def make_conf(pred, truth):
    conf = np.zeros_like(truth)
    conf[(pred==1)&(truth==1)] = 1
    conf[(pred==1)&(truth==0)] = 2
    conf[(pred==0)&(truth==1)] = 3
    return conf

for scene_id in TEST_SCENES:
    scene_chips = [c for c in test_chips if get_scene_id(c.name) == scene_id]
    positions   = []
    for c in scene_chips:
        parts = c.stem.split('_')
        try:
            row = int([p for p in parts if p.startswith('row')][0].replace('row',''))
            col = int([p for p in parts if p.startswith('col')][0].replace('col',''))
            positions.append(((row, col), c))
        except: continue
    if not positions:
        print(f'  Skipping {scene_id} — could not parse row/col')
        continue

    max_row = max(r for (r,_),_ in positions) + 512
    max_col = max(c for (_,c),_ in positions) + 512
    rgb_c   = np.zeros((max_row, max_col, 3), dtype=np.uint8)
    conf_n  = np.zeros((max_row, max_col), dtype=np.uint8)
    conf_b  = np.zeros((max_row, max_col), dtype=np.uint8)
    conf_e  = np.zeros((max_row, max_col), dtype=np.uint8)

    for (row, col), cp in positions:
        with rasterio.open(cp) as src:
            def s(a):
                p2, p98 = np.percentile(a, 2), np.percentile(a, 98)
                return np.clip((a-p2)/(p98-p2+1e-6)*255, 0, 255).astype(np.uint8)
            rgb_c[row:row+512, col:col+512, 0] = s(src.read(5).astype(np.float32))
            rgb_c[row:row+512, col:col+512, 1] = s(src.read(3).astype(np.float32))
            rgb_c[row:row+512, col:col+512, 2] = s(src.read(2).astype(np.float32))
        with rasterio.open(MASKS_DIR/cp.name) as src:
            truth = (src.read(1) > 0).astype(np.uint8)
        np_path = NDWI_MASKS_DIR / cp.name
        if np_path.exists():
            with rasterio.open(np_path) as src:
                conf_n[row:row+512, col:col+512] = make_conf(src.read(1), truth)
        bp = PROB_DIR_BEST / cp.name
        if bp.exists():
            with rasterio.open(bp) as src:
                conf_b[row:row+512, col:col+512] = make_conf(
                    (src.read(1) >= THRESH_BEST).astype(np.uint8), truth)
        ep = PRED_DIR_MAJORITY / cp.name
        if ep.exists():
            with rasterio.open(ep) as src:
                conf_e[row:row+512, col:col+512] = make_conf(src.read(1), truth)

    fig, axes = plt.subplots(1, 4, figsize=(28, 7))
    axes[0].imshow(rgb_c);                              axes[0].set_title('RGB');                            axes[0].axis('off')
    axes[1].imshow(conf_n, cmap=CMAP, vmin=0, vmax=3); axes[1].set_title(f'NDWI (t={NDWI_THRESH})');       axes[1].axis('off')
    axes[2].imshow(conf_b, cmap=CMAP, vmin=0, vmax=3); axes[2].set_title(f'U-Net best (t={THRESH_BEST})'); axes[2].axis('off')
    axes[3].imshow(conf_e, cmap=CMAP, vmin=0, vmax=3); axes[3].set_title('U-Net ensemble (majority vote)');axes[3].axis('off')
    fig.legend(handles=LEGEND, loc='lower center', ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.02))
    plt.suptitle(scene_id, fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig(CONF_DIR / f'{scene_id}.png', dpi=80, bbox_inches='tight')
    plt.close()
    print(f'  Saved: {scene_id}')

print(f'Done — {len(TEST_SCENES)} confusion maps saved to {CONF_DIR}')

In [ ]:
# ── Cell 10: Final Dissertation Summary ──

print('\n' + '='*70)
print('FINAL COMPARISON — All Methods on 4,788 Test Chips')
print('='*70)
print(summary_df.to_string(index=False))

# Final bar chart — Micro F1 only
fig, ax = plt.subplots(figsize=(12, 5))
COLORS = ['#2E75B6', '#5DCAA5', '#378ADD', '#1D9E75', '#B4B2A9', '#C8C8C8']
bars = ax.bar(summary_df['Method'], summary_df['Micro F1'], color=COLORS[:len(summary_df)])
for bar, val in zip(bars, summary_df['Micro F1']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylabel('Micro F1'); ax.set_ylim(0, 1.1)
ax.set_title('Final Comparison — Micro F1 on 4,788 Test Chips', fontsize=13, fontweight='bold')
plt.xticks(rotation=20, ha='right'); plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nSaved to: {OUTPUT_DIR}/final_comparison.csv')